# Phase 6: Recommendation Analysis

In this final notebook, we analyze the qualitative performance of our Hybrid Recommendation System. Beyond raw metrics like RMSE and MAP@10, we need to understand where our model succeeds, where it fails, and the inherent biases present in our data.


In [1]:
import pandas as pd
import numpy as np

# Load the SVD subset to analyze biases
df = pd.read_csv('../data/processed/netflix_svd_subset.csv')
titles_df = pd.read_csv('../data/raw/movie_titles.csv', encoding='ISO-8859-1', header=None, names=['movie_id', 'year', 'title'], on_bad_lines='skip')
title_map = dict(zip(titles_df['movie_id'], titles_df['title']))


## 1. Success and Failure Cases
In any recommendation system, predictions are never 100% perfect. Let's define what constitutes a success and a failure in our context.

### Success Cases
A **Success Case** occurs when the Hybrid model accurately captures the user's specific niche tastes rather than just recommending blockbusters. 
- **Example:** A user has exclusively given 5-star ratings to 1980s horror movies. The Item-CF component detects the high similarity between 80s horror films, while the SVD component maps the user's latent vector directly to the "classic horror" region. 
- **Result:** The system successfully outputs *The Evil Dead (1981)* with an estimated rating of 4.6, and the user subsequently rates it 5.0.

### Failure Cases
A **Failure Case** typically happens due to conflicting data signals or anomalous user behavior.
- **Example (The "Guilty Pleasure" Failure):** A user generally watches and highly rates prestigious Oscar-winning dramas. However, they watch one critically panned action movie and rate it 5 stars out of ironic enjoyment.
- **Result:** The SVD model assumes the user genuinely loves cheap action movies and begins flooding their Top-10 list with them, heavily skewing their previously accurate latent profile. The user rates these new recommendations 1-star, resulting in high prediction errors.


## 2. Popularity Bias Analysis
Collaborative filtering models are notoriously prone to popularity bias—recommending the most frequently rated movies to everyone. Let's quantify this bias in our dataset.

In [2]:
# Calculate the number of ratings per movie
movie_popularity = df.groupby('movie_id').size().reset_index(name='rating_count')
movie_popularity = movie_popularity.sort_values('rating_count', ascending=False)
movie_popularity['title'] = movie_popularity['movie_id'].map(title_map)

# Top 10 most rated movies
print("Top 10 Most Rated Movies:")
display(movie_popularity.head(10))

# Calculate the concentration of ratings
total_ratings = len(df)
top_100_ratings = movie_popularity.head(100)['rating_count'].sum()
concentration = (top_100_ratings / total_ratings) * 100

print(f"The top 100 movies account for {concentration:.2f}% of ALL ratings in our 7,291 movie catalog.")


Top 10 Most Rated Movies:


,movie_id,rating_count,title
760,1905,7338,Pirates of the Caribbean: The Curse of the Bla...
4612,11283,7230,Forrest Gump
1710,4306,7166,The Sixth Sense
5999,14691,7073,The Matrix
6167,15107,7067,Ocean's Eleven
6173,15124,7060,Independence Day
1131,2862,6958,The Silence of the Lambs
5879,14410,6956,Spider-Man
5590,13728,6875,Gladiator
2813,6971,6865,Ferris Bueller's Day Off


The top 100 movies account for 8.99% of ALL ratings in our 7,291 movie catalog.


**Impact on the Model:**
Because these top 100 movies account for nearly 9% of all interactions, the Item-CF similarity matrix can sometimes favor these familiar titles. This is one reason our Hybrid model uses a 70% SVD weight—to balance this natural popularity bias and allow the system to recommend more personalized, niche titles based on latent factors.

## 3. Cold-Start Problem
The "Cold-Start" problem refers to the system's inability to draw inferences for users or items with no historical data.

**User Cold-Start:**
If a brand new user joins the platform and has exactly 0 ratings, both SVD and Item-CF fail completely. The dot product for Item-CF resolves to 0, and SVD defaults to predicting the global dataset mean (~3.5 stars) for every single movie. 
- *Current Mitigation:* We filtered our dataset to users with >= 50 ratings, bypassing the issue entirely for training. In a production environment, we would need to implement a purely popularity-based fallback (like the Top 10 list above) until the new user rates at least 5 movies.

**Item Cold-Start:**
If a brand new movie is added to the catalog, it has 0 ratings. Collaborative filtering cannot recommend it to anyone because it has no similarity vectors or latent associations.


## 4. Future Improvements
To elevate this project to a production-ready enterprise system, we would need to address the structural limitations of collaborative filtering:

1. **Content-Based Filtering:** By incorporating movie metadata (e.g., Cast, Director, Genre, Plot Summary text), we could recommend new movies based on their attributes, entirely solving the Item Cold-Start problem.
2. **Deep Learning:** Implementing a neural collaborative filtering architecture (like Matrix Factorization integrated with Multi-Layer Perceptrons) could capture non-linear user-item interactions better than standard SVD.
3. **Temporal Dynamics:** User tastes change over time. Giving higher weights to movies rated in the last 6 months compared to movies rated 5 years ago would create a much more relevant recommendation list.
